In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install balance==0.23.0 fairlearn==0.14.0 -q

import random, os
SEED = 42
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds, TruePositiveRateParity
from balance import Sample
import warnings
warnings.filterwarnings('ignore')

print(f"scikit-learn: pandas {pd.__version__}, numpy {np.__version__}")
print("Setup complete")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.2/402.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 6.9 MB/s eta 0:00:00


INFO (2026-08-15 17:11:55,825) [__init__/<module> (line 77)]: Using balance version 0.23.0
INFO (2026-08-15 17:11:55,826) [__init__/<module> (line 82)]: 
balance (Version 0.23.0) loaded:
    📖 Documentation: https://import-balance.org/
    🛠️ Help / Issues: https://github.com/facebookresearch/balance/issues/
    📄 Citation:
        Sarig, T., Galili, T., & Eilat, R. (2023).
        balance - a Python package for balancing biased data samples.
        https://arxiv.org/abs/2307.06024

    Tip: You can view this message anytime with balance.help()



scikit-learn: pandas 2.2.2, numpy 2.0.2
Setup complete


In [3]:
col_names = ['checking_account','duration','credit_history','purpose',
'credit_amount','savings_account','employment_since','installment_rate',
'personal_status_sex','other_debtors','residence_since','property',
'age','other_installment_plans','housing','existing_credits','job',
'liable_people','telephone','foreign_worker','target']

df = pd.read_csv('/content/drive/MyDrive/german.data', sep=' ', header=None, names=col_names)
print(f"Raw dataset shape: {df.shape}")

df['target'] = df['target'].map({1: 1, 2: 0})
print(f"Target counts: {df['target'].value_counts().to_dict()}")

df['gender'] = df['personal_status_sex'].apply(lambda x: 1 if x in ['A92','A95'] else 0)
print(f"Female: {df['gender'].sum()} | Male: {(df['gender']==0).sum()}")

y = df['target']
gender = df['gender']
X = df.drop(['target','gender','personal_status_sex'], axis=1)
X = pd.get_dummies(X).astype(int)
X = X.reindex(sorted(X.columns), axis=1)
print(f"Features after encoding: {X.shape[1]}")

strat_var = gender.astype(str) + '_' + y.astype(str)
X_train, X_test, y_train, y_test, gender_train, gender_test = train_test_split(
    X, y, gender, test_size=0.3, random_state=SEED, stratify=strat_var)

y_train = y_train.values
y_test = y_test.values
gender_train = gender_train.values
gender_test = gender_test.values

print(f"\nTrain: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
print(f"Train female%: {gender_train.mean():.4f}")
print(f"Test female%: {gender_test.mean():.4f}")

X_train.to_csv('/content/drive/MyDrive/X_train_GITHUB.csv', index=False)
X_test.to_csv('/content/drive/MyDrive/X_test_GITHUB.csv', index=False)
pd.Series(y_train, name='target').to_csv('/content/drive/MyDrive/y_train_GITHUB.csv', index=False)
pd.Series(y_test, name='target').to_csv('/content/drive/MyDrive/y_test_GITHUB.csv', index=False)
pd.Series(gender_train, name='gender').to_csv('/content/drive/MyDrive/gender_train_GITHUB.csv', index=False)
pd.Series(gender_test, name='gender').to_csv('/content/drive/MyDrive/gender_test_GITHUB.csv', index=False)


Raw dataset shape: (1000, 21)
Target counts: {1: 700, 0: 300}
Female: 310 | Male: 690
Features after encoding: 57

Train: 700 rows | Test: 300 rows
Train female%: 0.3100
Test female%: 0.3100


In [4]:
# Stage 3: Pre-mitigation bias audit
# Measures how biased the training data is before any mitigation is applied

female_approval = y_train[gender_train == 1].mean()
male_approval = y_train[gender_train == 0].mean()
approval_gap = abs(male_approval - female_approval)

print("Approval Rate Gap")
print("Female approval rate:", round(female_approval*100, 2), "%")
print("Male approval rate:", round(male_approval*100, 2), "%")
print("Gap:", round(approval_gap, 4))

female_proportion = gender_train.mean()
representation_gap = abs(0.5 - female_proportion)

print("\nRepresentation Gap")
print("Female proportion:", round(female_proportion*100, 2), "%")
print("Gap:", round(representation_gap, 4))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
lr_proxy = LogisticRegression(solver='liblinear', max_iter=5000, random_state=SEED)
sauc_raw = cross_val_score(lr_proxy, X_train, gender_train, cv=cv, scoring='roc_auc').mean()
sauc_normalised = (sauc_raw - 0.5) * 2

print("\nProxy Feature Strength (sAUC)")
print("sAUC raw:", round(sauc_raw, 4))
print("sAUC normalised:", round(sauc_normalised, 4))

audit_score = (approval_gap * 0.5) + (representation_gap * 0.3) + (sauc_normalised * 0.2)

if audit_score <= 0.05:
    classification = "LOW"
elif audit_score <= 0.15:
    classification = "MODERATE"
else:
    classification = "HIGH"

print("\nPre-mitigation audit score:", round(audit_score, 4))
print("Classification:", classification)

Approval Rate Gap
Female approval rate: 64.98 %
Male approval rate: 72.26 %
Gap: 0.0728

Representation Gap
Female proportion: 31.0 %
Gap: 0.19

Proxy Feature Strength (sAUC)
sAUC raw: 0.6951
sAUC normalised: 0.3901

Pre-mitigation audit score: 0.1714
Classification: HIGH


In [5]:
# Stage 4: Balance weights using LASSO-IPW and CBPS

sample_df = X_train.copy()
sample_df['id'] = range(len(sample_df))
sample_df['gender'] = gender_train

male_df = sample_df[sample_df['gender'] == 0]
female_df = sample_df[sample_df['gender'] == 1]
female_oversampled = female_df.sample(n=len(male_df), replace=True, random_state=SEED)
target_df = pd.concat([male_df, female_oversampled], ignore_index=True)
target_df['id'] = range(len(target_df))

feature_cols = list(X_train.columns)
sample = Sample.from_frame(sample_df[['id'] + feature_cols], id_column='id')
target = Sample.from_frame(target_df[['id'] + feature_cols], id_column='id')
sample_with_target = sample.set_target(target)

print("Computing LASSO-IPW weights...")
lasso_weights = sample_with_target.adjust(method='ipw', max_de=1.5).weights().df.values.ravel()
print("Female mean weight:", round(lasso_weights[gender_train==1].mean(), 4))
print("Male mean weight:", round(lasso_weights[gender_train==0].mean(), 4))

print("\nComputing CBPS weights...")
cbps_weights = sample_with_target.adjust(method='cbps').weights().df.values.ravel()
print("Female mean weight:", round(cbps_weights[gender_train==1].mean(), 4))
print("Male mean weight:", round(cbps_weights[gender_train==0].mean(), 4))

def weighted_female_share(weights, gender):
    return np.sum(weights[gender == 1]) / np.sum(weights)

print("\nUnweighted female share:", round(gender_train.mean(), 4))
print("LASSO weighted female share:", round(weighted_female_share(lasso_weights, gender_train), 4))
print("CBPS weighted female share:", round(weighted_female_share(cbps_weights, gender_train), 4))

np.save('/content/drive/MyDrive/lasso_weights_GITHUB.npy', lasso_weights)
np.save('/content/drive/MyDrive/cbps_weights_GITHUB.npy', cbps_weights)


WARNING (2026-08-15 17:13:04,349) [sample_frame/from_frame (line 333)]: Casting id column to string
WARNING (2026-08-15 17:13:04,388) [pandas_utils/_warn_of_df_dtypes_change (line 519)]: The dtypes of SampleFrame._df were changed from the original dtypes of the input df, here are the differences - 
WARNING (2026-08-15 17:13:04,390) [pandas_utils/_warn_of_df_dtypes_change (line 530)]: The (old) dtypes that changed for df (before the change):
WARNING (2026-08-15 17:13:04,394) [pandas_utils/_warn_of_df_dtypes_change (line 533)]: 
job_A173                        int64
credit_history_A33              int64
housing_A153                    int64
employment_since_A73            int64
employment_since_A71            int64
other_debtors_A102              int64
other_debtors_A103              int64
property_A122                   int64
duration                        int64
existing_credits                int64
purpose_A44                     int64
purpose_A46                     int64
savings_acc

Computing LASSO-IPW weights...


INFO (2026-08-15 17:13:04,715) [adjustment/apply_transformations (line 472)]: Final variables in output: ['age', 'checking_account_A11', 'checking_account_A12', 'checking_account_A13', 'checking_account_A14', 'credit_amount', 'credit_history_A30', 'credit_history_A31', 'credit_history_A32', 'credit_history_A33', 'credit_history_A34', 'duration', 'employment_since_A71', 'employment_since_A72', 'employment_since_A73', 'employment_since_A74', 'employment_since_A75', 'existing_credits', 'foreign_worker_A201', 'foreign_worker_A202', 'housing_A151', 'housing_A152', 'housing_A153', 'installment_rate', 'job_A171', 'job_A172', 'job_A173', 'job_A174', 'liable_people', 'other_debtors_A101', 'other_debtors_A102', 'other_debtors_A103', 'other_installment_plans_A141', 'other_installment_plans_A142', 'other_installment_plans_A143', 'property_A121', 'property_A122', 'property_A123', 'property_A124', 'purpose_A40', 'purpose_A41', 'purpose_A410', 'purpose_A42', 'purpose_A43', 'purpose_A44', 'purpose_A45

Female mean weight: 1.4698
Male mean weight: 1.3396

Computing CBPS weights...


INFO (2026-08-15 17:13:30,437) [cbps/cbps (line 587)]: The formula used to build the model matrix: ['telephone_A192 + telephone_A191 + savings_account_A65 + savings_account_A64 + savings_account_A63 + savings_account_A62 + savings_account_A61 + residence_since + purpose_A49 + purpose_A48 + purpose_A46 + purpose_A45 + purpose_A44 + purpose_A43 + purpose_A42 + purpose_A410 + purpose_A41 + purpose_A40 + property_A124 + property_A123 + property_A122 + property_A121 + other_installment_plans_A143 + other_installment_plans_A142 + other_installment_plans_A141 + other_debtors_A103 + other_debtors_A102 + other_debtors_A101 + liable_people + job_A174 + job_A173 + job_A172 + job_A171 + installment_rate + housing_A153 + housing_A152 + housing_A151 + foreign_worker_A202 + foreign_worker_A201 + existing_credits + employment_since_A75 + employment_since_A74 + employment_since_A73 + employment_since_A72 + employment_since_A71 + duration + credit_history_A34 + credit_history_A33 + credit_history_A32 + 

Female mean weight: 1.489
Male mean weight: 1.331

Unweighted female share: 0.31
LASSO weighted female share: 0.3302
CBPS weighted female share: 0.3345


In [6]:
# Stage 5: Baseline models (no mitigation)

def evaluate_model(y_true, y_pred, gender):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    dpd = demographic_parity_difference(y_true, y_pred, sensitive_features=gender)
    eod = equalized_odds_difference(y_true, y_pred, sensitive_features=gender)
    female_mask = (gender == 1)
    female_true = y_true[female_mask]
    female_pred = y_pred[female_mask]
    actually_good_female = (female_true == 1)
    fnr_women = np.mean(female_pred[actually_good_female] == 0) if actually_good_female.sum() > 0 else np.nan
    return {'Accuracy': acc, 'Balanced_Accuracy': bal_acc, 'Sensitivity': sensitivity,
            'Specificity': specificity, 'DPD': dpd, 'EOD': eod, 'FNR_Women': fnr_women}

seeds = [42, 123, 456, 789, 1011]
lr_results = []
rf_results = []

for seed in seeds:
    lr = LogisticRegression(max_iter=5000, solver='liblinear', random_state=seed)
    lr.fit(X_train, y_train)
    lr_results.append(evaluate_model(y_test, lr.predict(X_test), gender_test))

    rf = RandomForestClassifier(random_state=seed)
    rf.fit(X_train, y_train)
    rf_results.append(evaluate_model(y_test, rf.predict(X_test), gender_test))

lr_avg = pd.DataFrame(lr_results).mean()
rf_avg = pd.DataFrame(rf_results).mean()

print("Logistic Regression baseline:")
print(lr_avg.round(4))
print("\nRandom Forest baseline:")
print(rf_avg.round(4))

baseline_df = pd.DataFrame({'Logistic_Regression': lr_avg, 'Random_Forest': rf_avg})
baseline_df.to_csv('/content/drive/MyDrive/baseline_results_GITHUB.csv')



Logistic Regression baseline:
Accuracy             0.7633
Balanced_Accuracy    0.6849
Sensitivity          0.8810
Specificity          0.4889
DPD                  0.0407
EOD                  0.0415
FNR_Women            0.1167
dtype: float64

Random Forest baseline:
Accuracy             0.7640
Balanced_Accuracy    0.6575
Sensitivity          0.9238
Specificity          0.3911
DPD                  0.0674
EOD                  0.1101
FNR_Women            0.0833
dtype: float64


In [7]:
# Stage 6: Group 1 - Fairlearn constraints alone, no Balance resampling

def get_constraint(name):
    if name == 'DP':
        return DemographicParity()
    elif name == 'EO':
        return EqualizedOdds()
    else:
        return TruePositiveRateParity()

group1_results = []
condition_num = 1

for constraint_name in ['DP', 'EO', 'EOP']:
    for clf_name in ['LR', 'RF']:
        accs, bal_accs, sens, specs, dpds, eods, fnrs = [], [], [], [], [], [], []
        for seed in seeds:
            if clf_name == 'LR':
                base = LogisticRegression(max_iter=5000, solver='liblinear', random_state=seed)
            else:
                base = RandomForestClassifier(random_state=seed)
            fair = ExponentiatedGradient(estimator=base, constraints=get_constraint(constraint_name), eps=0.01)
            fair.fit(X_train, y_train, sensitive_features=gender_train)
            pred = fair.predict(X_test)
            result = evaluate_model(y_test, pred, gender_test)
            accs.append(result['Accuracy'])
            bal_accs.append(result['Balanced_Accuracy'])
            sens.append(result['Sensitivity'])
            specs.append(result['Specificity'])
            dpds.append(result['DPD'])
            eods.append(result['EOD'])
            fnrs.append(result['FNR_Women'])

        row = {
            'Condition': condition_num, 'Balance': 'None', 'Constraint': constraint_name, 'Classifier': clf_name,
            'Accuracy': np.mean(accs), 'Balanced_Accuracy': np.mean(bal_accs),
            'Sensitivity': np.mean(sens), 'Specificity': np.mean(specs),
            'DPD': np.mean(dpds), 'EOD': np.mean(eods), 'FNR_Women': np.mean(fnrs)
        }
        group1_results.append(row)
        print(f"Condition {condition_num}: {constraint_name} + {clf_name}, DPD = {round(row['DPD'], 4)}")
        condition_num += 1

group1_df = pd.DataFrame(group1_results)
group1_df.to_csv('/content/drive/MyDrive/group1_results_GITHUB.csv', index=False)
print("\nGroup 1 saved")

Condition 1: DP + LR, DPD = 0.0227
Condition 2: DP + RF, DPD = 0.0707
Condition 3: EO + LR, DPD = 0.0395
Condition 4: EO + RF, DPD = 0.0674
Condition 5: EOP + LR, DPD = 0.0366
Condition 6: EOP + RF, DPD = 0.0951

Group 1 saved


In [9]:
group2_results = []
condition_num = 7

for constraint_name in ['DP', 'EO', 'EOP']:
    for clf_name in ['LR', 'RF']:
        accs, bal_accs, sens, specs, dpds, eods, fnrs = [], [], [], [], [], [], []
        for seed in seeds:
            base = LogisticRegression(max_iter=5000, solver='liblinear', random_state=seed) if clf_name == 'LR' else RandomForestClassifier(random_state=seed)
            fair = ExponentiatedGradient(estimator=base, constraints=get_constraint(constraint_name), eps=0.01)
            X_res, y_res, g_res = resample_with_weights(X_train, y_train, gender_train, lasso_weights, seed)
            fair.fit(X_res, y_res, sensitive_features=g_res)
            pred = fair.predict(X_test)
            result = evaluate_model(y_test, pred, gender_test)
            accs.append(result['Accuracy']); bal_accs.append(result['Balanced_Accuracy'])
            sens.append(result['Sensitivity']); specs.append(result['Specificity'])
            dpds.append(result['DPD']); eods.append(result['EOD']); fnrs.append(result['FNR_Women'])
        row = {'Condition': condition_num, 'Balance': 'LASSO', 'Constraint': constraint_name, 'Classifier': clf_name,
               'Accuracy': np.mean(accs), 'Balanced_Accuracy': np.mean(bal_accs),
               'Sensitivity': np.mean(sens), 'Specificity': np.mean(specs),
               'DPD': np.mean(dpds), 'EOD': np.mean(eods), 'FNR_Women': np.mean(fnrs)}
        group2_results.append(row)
        print(f"Condition {condition_num}: LASSO + {constraint_name} + {clf_name}, DPD = {round(row['DPD'], 4)}")
        condition_num += 1

group2_df = pd.DataFrame(group2_results)
group2_df.to_csv('/content/drive/MyDrive/group2_results_GITHUB.csv', index=False)
print("\nGroup 2 saved")

Condition 7: LASSO + DP + LR, DPD = 0.0338
Condition 8: LASSO + DP + RF, DPD = 0.0426
Condition 9: LASSO + EO + LR, DPD = 0.0337
Condition 10: LASSO + EO + RF, DPD = 0.0512
Condition 11: LASSO + EOP + LR, DPD = 0.0389
Condition 12: LASSO + EOP + RF, DPD = 0.0588

Group 2 saved


In [10]:
group3_results = []
condition_num = 13

for constraint_name in ['DP', 'EO', 'EOP']:
    for clf_name in ['LR', 'RF']:
        accs, bal_accs, sens, specs, dpds, eods, fnrs = [], [], [], [], [], [], []
        for seed in seeds:
            base = LogisticRegression(max_iter=5000, solver='liblinear', random_state=seed) if clf_name == 'LR' else RandomForestClassifier(random_state=seed)
            fair = ExponentiatedGradient(estimator=base, constraints=get_constraint(constraint_name), eps=0.01)
            X_res, y_res, g_res = resample_with_weights(X_train, y_train, gender_train, cbps_weights, seed)
            fair.fit(X_res, y_res, sensitive_features=g_res)
            pred = fair.predict(X_test)
            result = evaluate_model(y_test, pred, gender_test)
            accs.append(result['Accuracy']); bal_accs.append(result['Balanced_Accuracy'])
            sens.append(result['Sensitivity']); specs.append(result['Specificity'])
            dpds.append(result['DPD']); eods.append(result['EOD']); fnrs.append(result['FNR_Women'])
        row = {'Condition': condition_num, 'Balance': 'CBPS', 'Constraint': constraint_name, 'Classifier': clf_name,
               'Accuracy': np.mean(accs), 'Balanced_Accuracy': np.mean(bal_accs),
               'Sensitivity': np.mean(sens), 'Specificity': np.mean(specs),
               'DPD': np.mean(dpds), 'EOD': np.mean(eods), 'FNR_Women': np.mean(fnrs)}
        group3_results.append(row)
        print(f"Condition {condition_num}: CBPS + {constraint_name} + {clf_name}, DPD = {round(row['DPD'], 4)}")
        condition_num += 1

group3_df = pd.DataFrame(group3_results)
group3_df.to_csv('/content/drive/MyDrive/group3_results_GITHUB.csv', index=False)
print("\nGroup 3 saved")

Condition 13: CBPS + DP + LR, DPD = 0.0311
Condition 14: CBPS + DP + RF, DPD = 0.044
Condition 15: CBPS + EO + LR, DPD = 0.0214
Condition 16: CBPS + EO + RF, DPD = 0.0527
Condition 17: CBPS + EOP + LR, DPD = 0.0165
Condition 18: CBPS + EOP + RF, DPD = 0.0498

Group 3 saved


In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

def post_sauc(proba_scores, gender_test, seed=42):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    lr = LogisticRegression(solver='liblinear', max_iter=5000, random_state=seed)
    proba_2d = proba_scores.reshape(-1, 1)
    sauc_raw = cross_val_score(lr, proba_2d, gender_test, cv=cv, scoring='roc_auc').mean()
    return max((sauc_raw - 0.5) * 2, 0)

def get_proba_for_condition(balance_name, weights, constraint_name, clf_name, seed):
    base = LogisticRegression(max_iter=5000, solver='liblinear', random_state=seed) if clf_name == 'LR' else RandomForestClassifier(random_state=seed)
    if constraint_name is None:
        base.fit(X_train, y_train)
        return base.predict_proba(X_test)[:, 1]
    fair = ExponentiatedGradient(estimator=base, constraints=get_constraint(constraint_name), eps=0.01)
    if weights is None:
        fair.fit(X_train, y_train, sensitive_features=gender_train)
    else:
        X_res, y_res, g_res = resample_with_weights(X_train, y_train, gender_train, weights, seed)
        fair.fit(X_res, y_res, sensitive_features=g_res)
    return fair._pmf_predict(X_test)[:, 1]

def approval_gap_and_rep_gap(pred_binary):
    male_appr = pred_binary[gender_test == 0].mean()
    female_appr = pred_binary[gender_test == 1].mean()
    approval_gap = abs(male_appr - female_appr)
    approved_mask = (pred_binary == 1)
    female_among_approved = gender_test[approved_mask].mean()
    rep_gap = abs(0.5 - female_among_approved)
    return approval_gap, rep_gap

pre_mitigation_score = audit_score  # from Stage 3 earlier

conditions_to_audit = [
    ("Baseline LR", None, None, None, 'LR'),
    ("Condition 1", 'None', None, 'DP', 'LR'),
    ("Condition 15", 'CBPS', cbps_weights, 'EO', 'LR'),
    ("Condition 17", 'CBPS', cbps_weights, 'EOP', 'LR'),
]

print("Post-mitigation audit\n")

for label, bal_name, weights, constraint, clf in conditions_to_audit:
    sauc_list, appr_list, rep_list = [], [], []
    for seed in seeds:
        proba = get_proba_for_condition(bal_name, weights, constraint, clf, seed)
        pred_binary = (proba >= 0.5).astype(int)
        s = post_sauc(proba, gender_test, seed)
        a, r = approval_gap_and_rep_gap(pred_binary)
        sauc_list.append(s); appr_list.append(a); rep_list.append(r)

    mean_sauc = np.mean(sauc_list)
    mean_appr = np.mean(appr_list)
    mean_rep = np.mean(rep_list)
    score = (mean_appr * 0.5) + (mean_rep * 0.3) + (mean_sauc * 0.2)
    improvement = ((pre_mitigation_score - score) / pre_mitigation_score) * 100

    print(f"{label}: Approval Gap={round(mean_appr,4)} Rep Gap={round(mean_rep,4)} sAUC={round(mean_sauc,4)}")
    print(f"  Audit Score: {round(score,4)} | Improvement: {round(improvement,1)}%\n")

Post-mitigation audit

Baseline LR: Approval Gap=0.0407 Rep Gap=0.2013 sAUC=0.121
  Audit Score: 0.1049 | Improvement: 38.8%

Condition 1: Approval Gap=0.0061 Rep Gap=0.1883 sAUC=0.0119
  Audit Score: 0.0619 | Improvement: 63.9%

Condition 15: Approval Gap=0.0281 Rep Gap=0.1872 sAUC=0.009
  Audit Score: 0.072 | Improvement: 58.0%

Condition 17: Approval Gap=0.0249 Rep Gap=0.191 sAUC=0.015
  Audit Score: 0.0728 | Improvement: 57.6%

